In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_neo4j import Neo4jGraph
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser



d:\LANG CHAINS\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\LANG CHAINS\venv\lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [6]:

# ==========================================
# 1. SETUP GRAPH AND LLM
# ==========================================
# (Make sure your environment variables / passwords are correct)
NEO4J_URI = "neo4j+s://02410190.databases.neo4j.io"
NEO4J_USERNAME = "02410190"
NEO4J_PASSWORD = "i5_FbDv6SlusfdhmVkqPWcRaFiB_5RnXK_oiGx56gyM"
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)
llm = ChatGroq(model="llama3-8b-8192", temperature=0)



Unable to retrieve routing information


ValueError: Could not connect to Neo4j database. Please ensure that the url is correct

In [5]:

# ==========================================
# 2. HIGH ACCURACY PROMPT TEMPLATE
# ==========================================
# By passing the {schema} into the prompt, the LLM knows exactly what relationships and labels to use.
CYPHER_INSERT_TEMPLATE = """Task: Generate a Cypher statement to insert new data into the Neo4j graph.

Instructions:
1. Use only the provided relationship types and properties in the schema.
2. Do not use any other relationship types or properties that are not provided.
3. Generate only valid Cypher MERGE and SET statements. Always use MERGE to avoid creating duplicate nodes.
4. Return ONLY the raw Cypher query. Do not include any explanations, introductory text, or markdown formatting (like ```cypher).

Graph Schema:
{schema}

Data Description:
{description}
"""

cypher_prompt = PromptTemplate(
    input_variables=["schema", "description"],
    template=CYPHER_INSERT_TEMPLATE
)


In [ ]:

# ==========================================
# 3. CREATE THE INSERTION CHAIN
# ==========================================
insert_chain = cypher_prompt | llm | StrOutputParser()


In [ ]:

# ==========================================
# 4. GENERATE CYPHER AND EXECUTE
# ==========================================
data_description = (
    "A new movie called 'Interstellar' was released in 2014. It is a Sci-Fi movie. "
    "Matthew McConaughey acted in it as 'Cooper'. Christopher Nolan directed it."
)

print("Generating highly accurate Cypher based on Graph Schema...\n")

# Pass BOTH the dynamic schema and the natural language description
generated_cypher = insert_chain.invoke({
    "schema": graph.schema, 
    "description": data_description
})

# Clean up any potential markdown block backticks that the LLM might have output
generated_cypher = generated_cypher.replace("```cypher", "").replace("```", "").strip()

print(f"--- Generated Cypher Query ---\n{generated_cypher}\n")

# Execute it
print("Executing query...")
graph.query(generated_cypher)
graph.refresh_schema()
print("✅ Data successfully inserted and schema refreshed!")
